# Preparing Google Colab

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd drive/MyDrive/Datasets/bible-id

In [ ]:
ls

Install important library

In [ ]:
!pip install torch
!pip install transformers
!pip install sentence_transformers

In [ ]:
import pandas as pd
import numpy as np
import torch

In [ ]:
# Import necessary libraries
from sentence_transformers import SentenceTransformer, InputExample, losses, util
from torch.utils.data import DataLoader
import torch

# **IndoBerea**

**Tujuan:** menerapkan teknik pencarian semantik untuk menemukan ayat Alkitab yang relevan dengan kueri tertentu.

## Preparing Dataset

Kita perlu dataset yang berisi teks Alkitab dalam Bahasa Indonesia. Kita dapat memperolehnya dari berbagai sumber yang mempublikasikan teks Alkitab dalam berbagai bahasa.

Untuk penelitian ini, dataset yang digunakan diambil dari Kaggle yang bernama [Indonesian Bible oleh akun William Mulianto](https://www.kaggle.com/datasets/williammulianto/indonesia-bible-tb). Dataset ini merupakan hasil dari web-crawling pada website [alkitab.mobi](http://alkitab.mobi.com/).

Dataset ini terdiri dari dua versi Alkitab yang saling terpisah dalam format .csv, yaitu:
- Terjemahan Baru 1974
- Versi Mudah Dibaca

The purpose of fine-tuning here is to adjust the pre-trained model to our specific corpus, which is different from fine-tuning for a specific task such as classification or sentiment analysis. This process is more like unsupervised learning, where we don't have a specific target (labels) to learn from. The model learns by trying to predict the next word in a sentence, which helps it understand the context and semantics of the words in our specific corpus.

### Collecting



#### **JUST ONCE: COLLECT DATA FROM KAGGLE**

In [ ]:
from google.colab import files
files.upload()  # follow the prompt to upload your kaggle.json file

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d williammulianto/indonesia-bible-tb

#### **JUST FOR ONCE: UNZIPPING**

**get dataset from kaggle - just once**

In [ ]:
!kaggle datasets download -d williammulianto/indonesia-bible-tb

In [ ]:
ls

In [ ]:
!unzip "indonesia-bible-tb.zip"

#### Read Data

In [ ]:
tb = pd.read_csv('tb.csv')
vmd = pd.read_csv('vmd.csv')

In [ ]:
tb.head()

In [ ]:
vmd.head()

In [ ]:
print(tb.shape)
print(vmd.shape)

### Preprocessing

#### Concatenate columns: `kitab`, `pasal`, and `ayat`

In [ ]:
def reshape_data (dataset):

  # Convert integer columns to string
  dataset['pasal'] = dataset['pasal'].astype(str)
  dataset['ayat'] = dataset['ayat'].astype(str)

  # Combine three columns
  dataset['KITAB'] = dataset['kitab'] + ' ' + dataset['pasal'] + ':' + dataset['ayat']

  # Drop the original columns
  dataset.drop(['kitab', 'pasal', 'ayat', 'id'], axis=1, inplace=True)

  # Move column 'KITAB' to the first position
  dataset = dataset[['KITAB'] + [col for col in dataset.columns if col != 'KITAB']]

  return dataset

reshape `tb`

In [ ]:
new_tb = reshape_data(tb)
new_tb

save to csv

In [ ]:
new_tb.to_csv('new_tb.csv')

reshape `vmd`

In [ ]:
new_vmd = reshape_data(vmd)
new_vmd

save to csv

In [ ]:
new_vmd.to_csv('new_vmd.csv')

#### Combine `new_tb` and `new_vmd` into new dataframe

In [ ]:
# Merge new_tb and new_vmd on the 'KITAB' column
combine = new_tb.merge(new_vmd, on='KITAB', suffixes=('-new_tb', '-new_vmd'))

In [ ]:
combine

save to csv

In [ ]:
combine.to_csv('indonesian_bible.csv', index=False)



---



## Fine-Tuning Model

### Processing

In [ ]:
# Import necessary libraries
from sentence_transformers import SentenceTransformer, InputExample, losses, util
from torch.utils.data import DataLoader
import torch

In [ ]:
# Initialize the model
model = SentenceTransformer('indobenchmark/indobert-base-p2')

make sure all processing run in GPU

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

model = model.to(device)
model._first_module().auto_model.config.gradient_checkpointing = True

In [ ]:
import tensorflow as tf
tf.test.gpu_device_name()

In [ ]:
from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())

In [ ]:
# Preprocess the data
df['text'] = df['firman-new_tb'] + " " + df['firman-new_vmd']

# Convert the text data into a list of sentences
sentences = df['text'].tolist()

# Convert the list of sentences into a list of InputExample objects
examples = [InputExample(texts=[sentence]) for sentence in sentences]

The concatenation of the two columns, `df['firman-new_tb']` and `df['firman-new_vmd']`, into a new column `df['text']` is a simple way of creating a combined representation of the information in the two original columns for each row in the DataFrame.

In our case, we assume the `df['firman-new_tb']` and `df['firman-new_vmd']` columns contain different pieces of text information for each row. By concatenating these two columns, you're effectively creating a new text dataset that contains the complete context of each row.

The `SentenceTransformer` model expects a list of sentences (strings) as input. The `fit` method of the `SentenceTransformer` model doesn't accept pandas DataFrames or Series as input. Therefore, the data in the `df['text']` column is converted to a list of sentences with the `.tolist()` method.

After we have a list of sentences, we convert this list into a list of `InputExample` objects. Each `InputExample` object contains one or multiple texts (sentences), which will be encoded by the model into embeddings.

---

**NOTES:**

Please note that in this case, we are using `InputExample` objects with a single text (sentence). In other cases, like when training a model with pair-wise sentence similarity tasks, each `InputExample` would include a pair of sentences.

For example, if you were training a model to score the similarity between pairs of sentences, each `InputExample` would include a pair of sentences and a label indicating the similarity of the pair (such as a score from 0 to 1).

But in your case, as you are training the model for semantic search on a single text corpus, each `InputExample` includes a single sentence from your corpus.

---

The concatenation strategy essentially treats each verse as one single piece of information, regardless of whether it contains one sentence or multiple sentences. This can be beneficial because it maintains the context of the verse. In other words, all sentences in a verse are kept together as one unit, and are therefore processed together when the model generates an embedding for the verse.

There are several advantages to this approach:

1. **Maintaining context**: Some verses may contain multiple sentences that are closely related in meaning or theme. By treating the entire verse as one unit, you ensure that the relationships between these sentences are captured in the final embedding.

2. **Reducing computational load**: If each sentence within a verse were treated separately, the number of embeddings that the model needs to generate would increase, leading to a higher computational load.

However, the best strategy can vary depending on the specifics of your use case. If individual sentences within a verse are largely independent and you're interested in finding matches for sentence-level queries, it could make sense to treat each sentence as a separate unit during the embedding process.

It's also worth noting that the effectiveness of the concatenation strategy, like many aspects of machine learning, ultimately depends on the nature of your data and the specific task at hand. It's often a good idea to experiment with different preprocessing strategies and choose the one that works best for your particular project.

### Create The Model

#### With 1 Epochs

In [ ]:
# Convert the list of InputExample objects into a DataLoader
train_dataloader = DataLoader(examples, shuffle=True, batch_size=16)

# Define the training method
train_loss = losses.BatchHardTripletLoss(model=model)

# Train the model
model.fit(train_objectives=[(train_dataloader, train_loss)], epochs=1, warmup_steps=100)

The `InputExample` object is not designed to be used directly with the PyTorch `DataLoader`. The `SentenceTransformers` library provides a higher-level API to handle this.

To train the model, you should use the `model.fit` method directly with the list of `InputExample` objects and the loss function. The `DataLoader` conversion is handled internally by the `SentenceTransformers` library.

In [ ]:
model.save('IndoBerea-v2')

##### Using IndoBerea-v2 model

In [ ]:
model = SentenceTransformer('IndoBerea-v2')

In [ ]:
corpus = df['firman-new_tb'].to_list()

In [ ]:
# Generate embeddings for the sentences
sentence_embeddings = model.encode(corpus)

In [ ]:
# Define a query
query = "kasih Allah"

# Generate an embedding for the query
query_embedding = model.encode(query)

In [ ]:
# Perform semantic search
hits = util.semantic_search(query_embedding, sentence_embeddings)

print('Query:', query,'\n')

# Print the top 10 most similar sentences to the query
for hit in hits[0][:10]:
    print(corpus[hit['corpus_id']])

#### With 3 Epochs

In [ ]:
# Convert the list of InputExample objects into a DataLoader
train_dataloader = DataLoader(examples, shuffle=True, batch_size=16)

# Define the training method
train_loss = losses.BatchHardTripletLoss(model=model)

# Train the model
model.fit(train_objectives=[(train_dataloader, train_loss)], epochs=3, warmup_steps=100)

* Epochs: 3
* Training duration: 30 minutes
* GPU: Tesla T4 (15 GB)
* RAM: 12.7 GB

In [ ]:
model.save('IndoBerea-v3')

##Testing `IndoBerea-v3` model

In [ ]:
model3 = SentenceTransformer('IndoBerea-v3')

In [ ]:
combine.head()

In [ ]:
corpus = combine['firman-new_tb'].to_list()

In [ ]:
# Generate embeddings for the sentences
sentence_embeddings_3 = model3.encode(corpus)

### Testing 1

input query = kasih Allah

In [ ]:
# Define a query
query_3 = "kasih Allah"

# Generate an embedding for the query
query_embedding_3 = model3.encode(query_3)

In [ ]:
# Perform semantic search
hits_3 = util.semantic_search(query_embedding_3, sentence_embeddings_3)

print('Query:', query_3,'\n')

# Print the top 10 most similar sentences to the query
for hit in hits_3[0][:10]:
    print(corpus[hit['corpus_id']])

input query = kasih Allah pada manusia

In [ ]:
# Define a query
query_3 = "dosa manusia"

# Generate an embedding for the query
query_embedding_3 = model3.encode(query_3)

In [ ]:
# Perform semantic search
hits_3 = util.semantic_search(query_embedding_3, sentence_embeddings_3)

print('Query:', query_3,'\n')

# Print the top 10 most similar sentences to the query
for hit in hits_3[0][:10]:
    print(corpus[hit['corpus_id']])

####**Checking verse with identic sentences**

In [ ]:
from collections import Counter

In [ ]:
def count_duplicates(lst):
    ctr = Counter(lst)
    return {item: count for item, count in ctr.items() if count > 1}

In [ ]:
count_duplicates(corpus)

There are several verses that has identical sentences. For examples:
* *Keturunan Yafet ialah Gomer, Magog, Madai, Yawan, Tubal, Mesekh dan Tiras.* is written in Kejadian 10:2 and 1 Tawarikh 1:5

* *Jangan berzinah.* is written in Keluaran 20:14 and Ulangan 5:18

As a result, some verses are displayed multiple times. To address this, we will make some modifications when showing the result.

### Testing 2

* When model ordering some relevant verses, we will remove or exclude identic sentences. Then we will reorder it so that model keep showing 10 relevan verses to the query.

* We also want to see the books and the similarity score

In [ ]:
model3 = SentenceTransformer('IndoBerea-v3')

In [ ]:
corpus = df['firman-new_tb'].to_list()
books = df['KITAB'].to_list()

# Generate embeddings for the sentences
sentence_embeddings_3 = model3.encode(corpus)

Duration: 1m 16s

**Query** = *dosa manusia*

In [ ]:
# Define a query
query_3 = "dosa manusia"

# Generate an embedding for the query
query_embedding_3 = model3.encode(query_3)

Untuk menghapus duplikat dan tetap mendapatkan 10 hasil teratas yang unik, Anda bisa mengimplementasikan pendekatan berikut:

1. Pertama, lakukan semantic search seperti biasa dan dapatkan lebih banyak hasil daripada yang Anda butuhkan. Misalnya, jika Anda ingin 10 hasil unik, mungkin Anda perlu mendapatkan 20 atau 30 hasil teratas terlebih dahulu.
2. Lalu, iterasi hasil dari yang paling relevan ke yang paling tidak relevan, dan tambahkan ke list baru jika hasil tersebut belum ada di list.
3. Berhenti ketika Anda sudah memiliki 10 hasil unik.

Harap diperhatikan bahwa pendekatan ini mungkin masih memberikan kurang dari 10 hasil jika tidak ada cukup ayat unik dalam hasil teratas dari semantic search. Anda mungkin perlu menyesuaikan nilai `top_k` tergantung pada seberapa sering duplikat muncul dalam hasil Anda.

Untuk menampilkan Kitab, kita perlu menyimpan informasi kitab dan ayat dalam sebuah struktur data yang bisa dipetakan ke indeks yang sama dengan `sentence_embeddings`. Kita hanya perlu menambahkan informasi kitab ke dalam list `unique_hits` yang kita kumpulkan. Dengan cara ini, ketika kita mencetak hasilnya, kita akan mendapatkan informasi tentang kitab dan ayat untuk setiap hasil yang relevan.

In [ ]:
# Perform semantic search and get more results than required
hits_3 = util.semantic_search(query_embedding_3, sentence_embeddings_3, top_k=30)

# Initialize an empty list to store unique sentences
unique_hits = []

for hit in hits_3[0]:
    # Check if the sentence is already in the list
    if corpus[hit['corpus_id']] not in [existing_hit['corpus'] for existing_hit in unique_hits]:
        # If not, add it to the list
        unique_hits.append({'book': books[hit['corpus_id']], 'corpus': corpus[hit['corpus_id']], 'score': hit['score']})

    # Stop if we have 10 unique results
    if len(unique_hits) == 10:
        break

Dalam kode ini, kita tambahkan list `books` yang berisi informasi kitab dari DataFrame. Kemudian, ketika kita menambahkan hasil ke list `unique_hits`, kita juga menambahkan informasi kitab yang sesuai dengan ayat tersebut. Akhirnya, saat mencetak hasil, kita dapat mencetak informasi kitab dan ayat yang sesuai.

In [ ]:
print('Query:', query_3,'\n')

# Print the top 10 unique most similar sentences to the query
for hit in unique_hits:
    print(hit['book'], hit['corpus'], "(Score: {:.4f})".format(hit['score']))

### Testing 3

Good result. Now, we will do last modification:
* We will make a function. So that when we want to try using another query, we don't to write all code above

In [ ]:
def semantic_search(query, model, corpus, books, sentence_embeddings):
    # Generate an embedding for the query
    query_embedding = model.encode(query)

    # Perform semantic search and get more results than required
    hits = util.semantic_search(query_embedding, sentence_embeddings, top_k=30)

    # Initialize an empty list to store unique sentences
    unique_hits = []

    for hit in hits[0]:
        # Check if the sentence is already in the list
        if corpus[hit['corpus_id']] not in [existing_hit['corpus'] for existing_hit in unique_hits]:
            # If not, add it to the list
            unique_hits.append({'book': books[hit['corpus_id']], 'corpus': corpus[hit['corpus_id']], 'score': hit['score']})

        # Stop if we have 10 unique results
        if len(unique_hits) == 10:
            break

    print('QUERY:', query,'\n')
    print('RESULT:')

    # Print the top 10 unique most similar sentences to the query
    for hit in unique_hits:
        print(hit['book'], hit['corpus'], "(Score: {:.4f})".format(hit['score']))
    #return unique_hits

In [ ]:
corpus = combine['firman-new_tb'].to_list()
books = combine['KITAB'].to_list()

# Generate embeddings for the sentences
sentence_embeddings_3 = model3.encode(corpus)

#### Query 1: dosa manusia

In [ ]:
# Test the function
semantic_search("dosa manusia", model3, corpus, books, sentence_embeddings_3)

Untuk kueri "*dosa manusia*", model berhasil menghasilkan ayat-ayat yang memang berbicara tentang dosa dan perbuatan manusia, seperti "**Jangan berzinah**", "**Jangan membunuh**", dan "**Memikirkan kebodohan mendatangkan dosa, dan si pencemooh adalah kekejian bagi manusia**".

Meski ada beberapa ayat yang mungkin kurang relevan seperti "**Haruslah kaubuat ikat emas**", secara umum, hasilnya cukup memuaskan.

#### Query 2: kasih pada sesama

In [ ]:
# Test the function
semantic_search("kasih pada sesama", model3, corpus, books, sentence_embeddings_3)

Untuk kueri "*kasih pada sesama*", model juga berhasil menampilkan ayat-ayat yang berbicara tentang kasih, seperti "**Peliharalah kasih persaudaraan!**" dan "**Rahmat, damai sejahtera dan kasih kiranya melimpahi kamu**".

Meski ada beberapa ayat seperti "**Akulah roti hidup**" dan "**Haruslah kaubuat ikat emas**" yang mungkin kurang relevan, secara umum hasilnya cukup baik.

#### Query 3: pertobatan yang sejati

In [ ]:
# Test the function
semantic_search("pertobatan yang sejati", model3, corpus, books, sentence_embeddings_3)

#### Query 4: pengharapan dalam tekanan

In [ ]:
# Test the function
semantic_search("pengharapan dalam tekanan", model3, corpus, books, sentence_embeddings_3)

#### Query 5: tetap bersukacita

In [ ]:
# Test the function
semantic_search("tetap bersukacita", model3, corpus, books, sentence_embeddings_3)

#### Query 6: menantikan waktu Tuhan

In [ ]:
# Test the function
semantic_search("menantikan waktu Tuhan", model3, corpus, books, sentence_embeddings_3)

#### Query 7: mengampuni sesama

In [ ]:
# Test the function
semantic_search("mengampuni sesama", model3, corpus, books, sentence_embeddings_3)

#### Query 8: hidup kekal dan hasilnya

In [ ]:
# Test the function
semantic_search("hidup kekal dan hasilnya", model3, corpus, books, sentence_embeddings_3)

Hasil pencarian ini memberikan dasar berpikir bagi seseorang untuk menemukan makna dari setiap ayat tersebut berkaitan dengan kueri yang dimasukkan. Karena itu, selalu baik untuk memeriksa langsung dalam Alkitab kita sendiri untuk konteks dan interpretasi yang tepat.



---



Model semantik tidak sempurna dan akan selalu ada ruang untuk perbaikan. Berikut adalah beberapa saran yang bisa Anda lakukan dalam jangka waktu yang singkat:

1. **Optimalisasi Query**: Anda bisa mencoba untuk mengoptimalkan kueri Anda. Misalnya, Anda bisa mencoba menambahkan kata-kata atau frase yang lebih spesifik untuk membantu model dalam mencari ayat yang lebih relevan.

2. **Penyaringan Hasil**: Anda bisa mencoba untuk menyaring hasil pencarian semantik Anda. Misalnya, Anda bisa menyaring hasil yang memiliki skor kesamaan di bawah ambang batas tertentu atau hasil yang tidak mengandung kata-kata tertentu.

3. **Menggunakan Model yang Berbeda**: Jika Anda masih memiliki waktu, Anda bisa mencoba menggunakan model semantik yang berbeda. Misalnya, Anda bisa mencoba model seperti BERT atau RoBERTa yang telah dilatih pada teks berbahasa Indonesia.

4. **Mencoba Metode yang Berbeda**: Selain menggunakan model semantik, Anda bisa mencoba metode pencarian teks tradisional seperti TF-IDF atau BM25. Metode-metode ini mungkin tidak sekuat model semantik dalam menangkap makna teks, tetapi mereka bisa menjadi solusi yang baik jika Anda memiliki waktu yang terbatas.

Ingatlah bahwa Anda tidak perlu mengejar kesempurnaan dalam proyek ini. Tujuan Anda adalah untuk menunjukkan bahwa Anda bisa menerapkan teknik pencarian semantik untuk menemukan ayat Alkitab yang relevan dengan kueri tertentu. Selama Anda bisa melakukan itu, Anda sudah berhasil dalam proyek ini.

Untuk meningkatkan kemampuan model dalam menangkap nuansa kontekstual dari kueri, berikut beberapa hal yang bisa Anda coba:

1. **Jumlah Epochs**: Meningkatkan jumlah epochs dalam proses training bisa membantu model untuk belajar lebih banyak dari data. Namun, Anda harus berhati-hati untuk tidak overfitting.

2. **Ukuran Batch**: Mengubah ukuran batch juga bisa mempengaruhi hasil. Ukuran batch yang lebih besar bisa mengurangi noise dalam update gradient, namun bisa memerlukan lebih banyak memori dan waktu komputasi.

3. **Loss Function**: Anda saat ini menggunakan `BatchHardTripletLoss`. Anda bisa mencoba loss function lain seperti `CosineSimilarityLoss` atau `MultipleNegativesRankingLoss`.

6. **Fine-Tuning dengan Data Spesifik Domain**: Jika memungkinkan, Anda bisa mencoba melakukan fine-tuning model Anda dengan dataset spesifik domain yang lebih mirip dengan data yang Anda coba cari.

7. **Penggunaan Contextualized Embeddings**: Embedding kontekstual seperti yang dihasilkan oleh model Transformer dapat membantu menangkap makna kata dalam konteksnya, yang bisa sangat berguna untuk pencarian semantik.

Perlu diingat bahwa setiap perubahan yang Anda buat harus diuji untuk memastikan bahwa mereka benar-benar memberikan peningkatan kinerja. Anda mungkin perlu melakukan beberapa eksperimen untuk menemukan pengaturan terbaik untuk kasus Anda.

3. **Loss Function**:

   - `CosineSimilarityLoss`: Loss ini mengukur dissimilarity antara dua vektor dengan mencari cosine dari sudut antara kedua vektor tersebut. Loss ini baik digunakan ketika Anda ingin mengukur sejauh mana dua vektor menunjuk ke arah yang sama.

   - `MultipleNegativesRankingLoss`: Loss ini merupakan variasi dari triplet loss. Dalam kasus ini, untuk setiap vektor positif, beberapa vektor negatif dipilih dan loss dihitung berdasarkan peringkat vektor negatif tersebut. Loss ini baik digunakan ketika Anda memiliki banyak contoh negatif.

4. **Pilihan Model**: Anda benar, model `'indobenchmark/indobert-base-p2'` mungkin lebih cocok untuk kasus Anda karena model tersebut dilatih dengan teks bahasa Indonesia. Model baseline milik SentenceTransformers memang kebanyakan dilatih dengan data berbahasa Inggris. Namun, perlu diingat bahwa tidak semua model yang dilatih dengan data dalam bahasa tertentu akan memberikan performa yang baik dalam semua tugas dalam bahasa tersebut. Performa model sangat bergantung pada sifat tugas dan data spesifik yang digunakan.